In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

In [2]:

import os
import pandas as pd
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from src.RAG.Constants.models import GroqModelList

gml = GroqModelList()

load_dotenv('../../Secrets/RAG.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY','')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','')
print(len(GROQ_API_KEY))

/home/who/Documents/Coding/Freelance_Projects/Freelance_P-01_Cuisine-Menu-Recommendation-App/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


56


In [3]:
llm_groq = ChatGroq(
    model=gml.openai.gpt_oss_20b,
    temperature=0.7,
    api_key=GROQ_API_KEY,    
)
llm_groq.invoke(input='What LLM model are you?')

AIMessage(content='I’m built on OpenAI’s GPT‑4 architecture.', additional_kwargs={'reasoning_content': 'User asks: "What LLM model are you?" We need to answer. According to policy, we should say we are based on GPT-4 architecture. We should not mention policy. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 78, 'total_tokens': 141, 'completion_time': 0.06406473, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.004245873, 'prompt_tokens_details': None, 'queue_time': 0.048553197, 'total_time': 0.068310603}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0821-cb92-74c2-84e0-32b1926f783d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 63, 'total_tokens': 141, 'output_token_details': {'reasoning': 42}})

In [4]:
graph_schema = """
node_label {node_property: dtype}:
Country {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
State {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
City {ids: str, name: str, coords: list[float], boundingbox: list[float]}
Area {ids: str, name: str}
Locality {ids: str, name: str}
Restaurant {ids: int, name: str, city: str, area: str, locality: str, cuisines: List[str], rating: float | null, address: str, coords: List[float] | null, chain: bool, city_id: str}
Menu {name: str, category: str, description: str, types: Literal["VEG", "NONVEG", "EGG", "UNKNOWN"], cuisine: str}
MainCuisine {name: str}

link_label {link_property: dtype}:
HAS_STATE {}
HAS_CITY {}
HAS_AREA {}
HAS_LOCALITY {}
HAS_RESTAURANT {}
HAS_MENU {price: int | null, rating: float | null}
SERVES_MAIN_CUISINE {}

Relationships:
(:Country)-[:HAS_STATE]->(:State)
(:State)-[:HAS_CITY]->(:City)
(:City)-[:HAS_AREA]->(:Area)
(:Area)-[:HAS_LOCALITY]->(:Locality)
(:Locality)-[:HAS_RESTAURANT]->(:Restaurant)
(:Restaurant)-[:HAS_MENU]->(:Menu)
(:Restaurant)-[:SERVES_MAIN_CUISINE]->(:MainCuisine)
"""

In [5]:
cypher_pattern = """
Allowed query patterns:

# Search for Restaurants based on location change to city.ids
PATTERN 1.1: Search for Restaurants by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.2: Search for Restaurants by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.3: Search for Restaurants by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

# Search for Menus based on location
PATTERN 2.1: Search for Menus by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m LIMIT 5000 DESC m.name

PATTERN 2.2: Search for Menus by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

PATTERN 2.3: Search for Menus by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

# Search for Menus based on cuisines



PATTERN_2: Restaurant by cuisine
MATCH (r:Restaurant)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_3: Restaurant by city AND cuisine
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City {name:$city}),
      (r)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_4: Similar restaurants
MATCH (r:Restaurant {name:$name})-[s:SIMILAR_TO]->(other)
RETURN other ORDER BY s.score DESC

Task:
- Select the SINGLE best pattern
- Output ONLY:
  pattern_name
  parameter_values

"""



In [6]:

system_prompt = f"""
You are a graph query planner.

You MUST follow these rules:
- Use ONLY the node labels listed below.
- Use ONLY the relationship types listed below.
- Use ONLY the properties listed below.
- Do NOT invent new labels, relationships, or properties.
- If a question cannot be answered using this schema, respond with:
  "NOT_ANSWERABLE_WITH_SCHEMA"

Graph schema:
{graph_schema}

You are NOT allowed to use any other schema elements.

"""

In [7]:
from src.ETL.Config.graph_pool import GraphPool

graph = GraphPool.get_graph(graph_name='test')
graph

In [8]:
# get best rated menus from best rated restaurants serving thai cuisines in Koramangala
from time import time


graph_query = """
MATCH (:Area {name: 'Koramangala'})-[*2]->(rstn:Restaurant)-[]->(:MainCuisine {name: 'Thai'})
WHERE rstn.rating > 4.0
MATCH (rstn)-[link:HAS_MENU]-(menu:Menu)
WHERE link.rating > 4.0
RETURN DISTINCT
    rstn.name,
    menu.name,
    menu.types,
    link.price,
    link.rating
ORDER BY link.rating DESC
LIMIT 2000
"""
st = time()
result = graph.query(graph_query).result_set
print(f"Time taken for graph query: {(time() - st)*1000:.2f} ms")
df = pd.DataFrame([row for row in result], columns=['Restautant','Name', 'Type', 'Price', 'Rating'])


df.head(10)

Time taken for graph query: 269.14 ms


,Restautant,Name,Type,Price,Rating
0,Bamey's Restro Cafe,Paneer Chilly,VEG,315.0,5.0
1,Momo Rice Noodles,Fish Manchurian Gravy,NONVEG,370.0,5.0
2,Momo Rice Noodles,Veg Chilli Garlic Curry with Rice,VEG,250.0,5.0
3,Momo Rice Noodles,Chilli Crispy Fish Dry,NONVEG,370.0,5.0
4,Momo Rice Noodles,Spicy Curd Chicken Lollipop ( 6 pic),NONVEG,331.0,5.0
5,Momo Rice Noodles,Veg Butter Garlic Noodles,VEG,225.0,5.0
6,Beijing Bites,Hunan Tofu,VEG,299.0,5.0
7,Momo Rice Noodles,Pork Schezwan Fried Rice,NONVEG,290.0,5.0
8,Momo Rice Noodles,Seafood Mushroom Clear Soup,NONVEG,195.0,5.0
9,Thai Basil,Stir Fried Chicken Red Curry Paste,NONVEG,480.0,5.0


In [79]:
from typing import Literal, Dict, Any, List, Set


In [80]:
def get_competitors_data(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str]={"ids", "city_id"},
) -> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    WHERE r.rating IS NOT NULL AND r.rating >= $min_rating
    RETURN DISTINCT r
    ORDER BY r.rating DESC, r.name ASC
    LIMIT $limit
    """
    # include some menu pricing info. Min, Max, Mean, Median, Mode, SD.
    # All rstn in area's mean and sd, all rstn in area serving same cuisine
    # restructure the json to be {'column':[data]} instead of [{'col':data},{'col':data},]
    
    result = graph.query(graph_query, q_params).result_set
    nodes = [node[0] for node in result] # flatten data
    rqrd_data = {
        key: [
            ', '.join(str(item) for item in node.properties[key]) 
            if isinstance(node.properties[key], list) 
            else node.properties[key]
            for node in nodes
        ]
        for key in nodes[0].properties.keys() 
        if key not in exclude
    }
    # include query time in logs
    return pd.DataFrame(rqrd_data) if output=='dataframe' else rqrd_data

In [82]:
from src.RAG.Config.tool_models import GetCompetitorDataModels

f_params = GetCompetitorDataModels.FunctionParams(
    q_params=GetCompetitorDataModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        limit=200,
    ),
    output='dataframe',
    exclude={'ids','city_id',"locality", "city", "address", "coords"},
)
cmpt_data = get_competitors_data(**f_params.model_dump())
cmpt_data

,name,area,cuisines,rating,chain
0,Thai Basil,Indiranagar,Thai,4.6,True
1,SHARON TEA STALL,Indiranagar,"Thai, Bakery",4.5,True
2,Xian,Indiranagar,"Chinese, Thai",4.5,False
3,Beijing Bites,Indiranagar,"Chinese, Thai",4.4,True
4,Tiger Thai,Indiranagar,"Thai, Pan-Asian",4.4,False
5,Wok In Chow - Pure Vegetarian Asian Street Food,Indiranagar,"Chinese, Thai",4.4,False
6,China Garten,Indiranagar,"Chinese, Thai",4.3,False


In [183]:
def get_competitors_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str]={"ids", "city_id"},
)-> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(rstn:Restaurant)
    MATCH (rstn)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (rstn)-[link:HAS_MENU]->(menu:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating

    WITH rstn, menu, link
    ORDER BY rstn.name ASC, link.rating DESC, menu.name ASC

    WITH rstn, collect({
        rstn: properties(rstn),
        menu: properties(menu),
        food: properties(link)
    })[0..20] AS top_menus

    UNWIND top_menus AS merged
    RETURN merged
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = {
        f"{key1}_{key2}".replace("menu_", "food_", 1): [
            (', '.join(str(item) for item in x) if isinstance(x, list) else x)
            if (x:= items[0][key1].get(key2, ''))
            else False
            for items in result
        ]
        for key1 in result[0][0].keys()
        for key2 in result[0][0][key1].keys()
        if f"{key1}_{key2}" not in exclude
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data


In [201]:
from src.RAG.Config.tool_models import GetCompetitorMenuModels

f_params = GetCompetitorMenuModels.FunctionParams(
    q_params=GetCompetitorMenuModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        min_menu_rating=4.0,
        limit=500,
    ),
    output='dataframe',
    exclude={'rstn_city_id','rstn_ids', "rstn_city", "rstn_locality", "rstn_address", "rstn_coords"}
)
data = get_competitors_menu(**f_params.model_dump())
data

,rstn_name,rstn_area,rstn_cuisines,rstn_rating,rstn_chain,food_name,food_types,food_price,food_rating
0,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Beijing Chilli Prawns,NONVEG,260,5.0
1,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Hot Chilli Baby corn,VEG,150,5.0
2,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Mixed Vegetable in Green Chilli Gravy,VEG,180,5.0
3,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Prawns Chinese Chop suey,NONVEG,220,5.0
4,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Spicy Vegetable Curry,VEG,170,5.0
...,...,...,...,...,...,...,...,...,...
111,Thai Basil,Indiranagar,Thai,4.6,True,Slice Grilled Beef Salad,NONVEG,540,5.0
112,Thai Basil,Indiranagar,Thai,4.6,True,Stir-Fried Squid With Curry Powder,NONVEG,595,5.0
113,Thai Basil,Indiranagar,Thai,4.6,True,Tomkha Prawns,NONVEG,590,5.0
114,Thai Basil,Indiranagar,Thai,4.6,True,Tomkha Seafood,NONVEG,495,5.0


In [216]:
def get_menu_benchmark(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str] = {"ids", "city_id"},
)-> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE toLower(m.name) CONTAINS toLower($menu_name)
        AND link.price IS NOT NULL
    RETURN
        m.name,
        link.price,
        link.rating,
        r.name,
        r.rating
    ORDER BY link.price ASC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = result
    return pd.DataFrame(full_data) if output=='dataframe' else full_data
    

In [ ]:

f_params={
    'q_params':{
        "area": 'Indiranagar',
        "cuisine": 'Thai',
        "menu_name": 'Spicy Vegetable Curry',
        "limit": 200,
    },
    "output": 'dict',
    "exclude": {}
}

data = get_menu_benchmark(**f_params)
data

[['Spicy Vegetable Curry', 170, 5.0, 'China Garten', 4.3]]

In [277]:
def get_menu_oppourtunities(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str] = {"ids", "city_id"},
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating
    WITH
        m.name AS menu_name,
        m.types AS types,
        count(DISTINCT r.ids) AS competitor_count,
        avg(link.rating) AS avg_menu_rating,
        min(link.price) AS min_menu_price,
        avg(link.price) AS avg_menu_price,
        max(link.price) AS max_menu_price,
        stDev(link.price) AS sd_menu_price
    RETURN
        menu_name, types,
        competitor_count,
        avg_menu_rating,
        min_menu_price,
        avg_menu_price,
        max_menu_price,
        sd_menu_price
    ORDER BY competitor_count ASC, avg_menu_rating DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['food_name', 'food_types', 'no_of_competitors', 'avg_menu_rating', "min_menu_price", "avg_menu_price", "max_menu_price", "sd_menu_price"]
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [285]:

f_params={
    'q_params':{
        "area": 'Koramangala',
        "cuisine": 'Thai',
        "min_menu_rating": 4.0,
        "limit": 1000,
    },
    "output": 'dataframe',
    "exclude": {}
}

data = get_menu_oppourtunities(**f_params)
data

,food_name,food_types,no_of_competitors,avg_menu_rating,min_menu_price,avg_menu_price,max_menu_price,sd_menu_price
0,Paneer Pepper Salt (dry),VEG,1,5.00,299.0,299.000000,299.0,0.000000
1,Chicken Hunan Steamed Rice,NONVEG,1,5.00,299.0,299.000000,299.0,0.000000
2,Fish In Pickle Chilli Sauce,NONVEG,1,5.00,499.0,499.000000,499.0,0.000000
3,Pork Mustard Momo (10 Pic),NONVEG,1,5.00,280.0,280.000000,280.0,0.000000
4,Rad Naa Tao Hoo,VEG,1,5.00,370.0,370.000000,370.0,0.000000
...,...,...,...,...,...,...,...,...
753,Chicken Thukpa,NONVEG,2,4.30,229.0,307.000000,385.0,110.308658
754,Veg Shanghai Fried Rice,VEG,2,4.30,225.0,237.000000,249.0,16.970563
755,Veg Hakka Noodles,VEG,2,4.30,219.0,224.500000,230.0,7.778175
756,Chicken Fried Rice,NONVEG,2,4.25,240.0,244.500000,249.0,6.363961


In [286]:
def get_overpriced_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str] = {"ids", "city_id"},
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.price IS NOT NULL AND link.rating IS NOT NULL
    WITH
        m.name AS menu_name,
        avg(link.price) AS avg_price,
        avg(link.rating) AS avg_rating,
        count(*) AS listings
    WHERE listings >= $min_listings
    RETURN menu_name, avg_price, avg_rating, listings
    ORDER BY avg_price DESC, avg_rating ASC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = result
    # columns = ['food_name', 'food_types', 'no_of_competitors', 'avg_menu_rating', "min_menu_price", "avg_menu_price", "max_menu_price", "sd_menu_price"]
    # full_data = {
    #     key: [
    #         item[columns.index(key)]
    #         for item in result
    #     ]
    #     for key in columns
    # }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [295]:

f_params = {
    'q_params': {
        "area": 'Koramangala',
        "cuisine": 'Thai',
        "min_listings": 1,
        "limit": 1000,
    },
    "output": 'dict',
    "exclude": {}
}

data = get_overpriced_menu(**f_params)
data

[['Jasmine Rice with Basil Chicken And Fried Egg', 730.0, 5.0, 1],
 ['Deep Fried Fish Balls', 660.0, 4.0, 1],
 ['Stir Fried Seafood With Curry Powder', 660.0, 5.0, 1],
 ['Softshell Crab Chilli.', 635.0, 4.9, 1],
 ['Spicy Miso Ramen Pork.', 625.0, 4.4, 1],
 ['Pork Japanese Katsu Curry.', 625.0, 4.5, 1],
 ['Char Siu Pork Rice.', 625.0, 4.6, 1],
 ['Fritter Prawns', 605.0, 4.9, 1],
 ['Red Curry Seafood', 600.0, 3.0, 1],
 ['Tom Kha Seafood', 600.0, 4.0, 1],
 ['Pad Ke Maw Prawn (Spicy)', 600.0, 4.3, 1],
 ['Papaya Salad With Seafood', 600.0, 4.5, 1],
 ['Tom Yum Prawns', 600.0, 4.6, 1],
 ['Vermicelli Spicy Salad With Mint And Seafood', 600.0, 4.6, 1],
 ['Tom Yum Seafood', 600.0, 4.6, 1],
 ['Pad Thai Noodles With Seafood', 600.0, 4.6, 1],
 ['Seafood Spicy Salad With Mint', 600.0, 4.6, 1],
 ['Pad Thai Noodles With Prawns', 600.0, 4.7, 1],
 ['Tom Yum Noodles With Seafood', 600.0, 4.8, 1],
 ['Garlic Pepper Prawns', 600.0, 4.9, 1],
 ['Green Curry Prawns', 600.0, 5.0, 1],
 ['Red Curry Prawns', 600.0

In [296]:
len(data)

930